# Chấm COMET cho adapter LoRA chung Hoa ↔ Việt
Chọn runtime **T4** rồi **Run all**, cấp quyền Drive khi được hỏi. Notebook đọc kết quả test đã có, không train và không dịch lại.

Cần có `score_comet.py` ở thư mục gốc My Drive. Điểm ghi vào `artifacts/comet-test-zh-vi` và `artifacts/comet-test-vi-zh` của run.

Chạy lại an toàn: chiều đã chấm xong sẽ được bỏ qua. Khi xong, chọn **Runtime → Disconnect and delete runtime**.


In [ ]:
from google.colab import drive
from hashlib import sha256
import json, subprocess
from pathlib import Path

drive.mount("/content/drive")
RUN = Path("/content/drive/MyDrive/DACNTT-lora/run-all/shared-lora-a100-40-t4-e4-901c4fc12e")
SCRIPT = Path("/content/drive/MyDrive/score_comet.py")
SCRIPT_SHA256 = "f422ce62f2966d5ab3e488e9546c755a8f8abdb4bbcf10e36900a6b32ecb0729"
DIRECTIONS = ("zh-vi", "vi-zh")
PY = "/content/.venv-comet/bin/python"

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError("Chưa có GPU: chọn Runtime → Change runtime type → T4.")
print("GPU:", gpu.stdout.strip())
if not SCRIPT.is_file():
    raise FileNotFoundError(f"Thiếu {SCRIPT}")
if sha256(SCRIPT.read_bytes()).hexdigest() != SCRIPT_SHA256:
    raise ValueError("score_comet.py trên Drive khác bản đã đóng gói với notebook này.")
for direction in DIRECTIONS:
    folder = RUN / "artifacts" / f"full-test-{direction}"
    missing = [n for n in ("sources.jsonl", "predictions.jsonl", "references.jsonl", "metrics.json") if not (folder / n).is_file()]
    if missing:
        raise FileNotFoundError(f"{folder} thiếu {missing}")
print("Đầu vào đủ cho hai chiều.")


In [ ]:
!pip install -q uv
!uv venv --python 3.11 --allow-existing /content/.venv-comet
!uv pip install -q --python /content/.venv-comet/bin/python unbabel-comet==2.2.7 torch==2.8.0 numpy==1.26.4 transformers==4.57.6 pytorch-lightning==2.6.6 torchmetrics==0.10.3 setuptools==78.1.0


In [ ]:
BATCH_SIZE = 32  # Hết bộ nhớ GPU thì đổi thành 16 rồi chạy lại ô này

for direction in DIRECTIONS:
    source = RUN / "artifacts" / f"full-test-{direction}"
    output = RUN / "artifacts" / f"comet-test-{direction}"
    if (output / "comet_metrics.json").is_file():
        print("Đã chấm xong, bỏ qua:", direction)
        continue
    if output.exists():
        raise FileExistsError(f"{output} còn dở từ lần trước. Kiểm tra thư mục này, không tự xóa khi chưa xem.")
    print("Đang chấm:", direction)
    !{PY} {SCRIPT} --eval-dir "{source}" --output-dir "{output}" --batch-size {BATCH_SIZE}
    if not (output / "comet_metrics.json").is_file():
        raise RuntimeError(f"Chấm {direction} không thành công; xem thông báo lỗi phía trên.")


In [ ]:
for direction in DIRECTIONS:
    comet = json.loads((RUN / "artifacts" / f"comet-test-{direction}" / "comet_metrics.json").read_text(encoding="utf-8"))
    base = json.loads((RUN / "artifacts" / f"full-test-{direction}" / "metrics.json").read_text(encoding="utf-8"))
    print(f"{direction}: BLEU {base['bleu']:.2f} | chrF {base['chrf']:.2f} | COMET {comet['comet']:.4f} | "
          f"{comet['count']} câu | chuẩn hóa {comet['text_normalization']} | {comet['elapsed_seconds']:.0f} giây")
print("Xong. Hãy chọn Runtime → Disconnect and delete runtime.")
